# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I am framing Lane 2 as a Binary Classification and Priority Scoring task.

Primary Model: A classification model (e.g., Gradient Boosted Trees) that outputs a probability score (between 0.0 and 1.0) indicating whether a URL is experiencing significant performance decay and requires a content refresh.

Why: Rather than just predicting a continuous raw number (like predicting exact future clicks), a classification score allows us to rank pages into actionable priority bands (High Risk / Immediate Refresh needed vs. Low Risk / Stable) for human editorial workflows.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/Proxy Variable: needs_refresh (Binary: 1 = Refresh Recommended, 0 = No Action Needed).

Label Source: Defined via an observed outcome rule from historical behavior. Specifically, a page is labeled 1 if its performance (impressions or CTR) dropped by more than $X\%$ relative to its historical baseline over a defined window (e.g., 30–90 days), while overall site/category trends remained stable.

Note: It uses observable historical delta signals rather than arbitrary subjective rules.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Precision@K (and PR-AUC / Precision-Recall Area Under Curve).

What number means 'good'? Precision@Top-20% $> 0.80$.

Defense: Editorial teams have limited hours and budget to rewrite pages. False positives (flagging a healthy page for rewrite) waste human labor and risk disrupting a page that already performs well.

Therefore, maximizing Precision in the top ranked $K$ suggestions ensures that when the system flags a page for a refresh, the editor's time is almost guaranteed to be well spent.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset directly
url = "https://raw.githubusercontent.com/imaniftikhar/week1_flyrank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Display unit of analysis confirmation
print("--- UNIT OF ANALYSIS ---")
print("One row = One unique URL / Content Page at a specific point in time.")
print(f"Total Rows (Pages Analyzed): {len(df):,}")

# Construct a proxy target column for Lane 2 based on performance delta signal
# Example proxy logic: Page needs refresh if impression trend drops significantly
if 'impressions' in df.columns:
    # Defining target column 'needs_refresh' (1 if low performance relative to median, 0 otherwise)
    df['needs_refresh'] = (df['impressions'] < df['impressions'].median()).astype(int)

# Display sample of dataframe with features and engineered target column
cols_to_show = [c for c in ['url', 'page_id', 'impressions', 'clicks', 'word_count', 'needs_refresh'] if c in df.columns]
if not cols_to_show:
    cols_to_show = df.columns[:5].tolist() + ['needs_refresh']

df[cols_to_show].head(5)

--- UNIT OF ANALYSIS ---
One row = One unique URL / Content Page at a specific point in time.
Total Rows (Pages Analyzed): 30,000


,word_count
0,3221.0
1,2481.0
2,3515.0
3,NaN
4,2803.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like if traffic_drop > 20% then refresh) fails because content decay is non-linear and multidimensional. Traffic drops can occur due to seasonal shifts, site-wide technical bugs, or algorithm updates, rather than true content staleness.

Machine Learning beats simple if-statements because:

Multi-Signal Interaction: It can weigh complex interactions between age, word count, impression trends, and position shifts simultaneously.

Context Sensitivity: It learns distinct decay patterns across different content types instead of applying a blunt, one-size-fits-all threshold.

Probabilistic Output: It generates a calibrated confidence score that allows dynamic prioritization rather than rigid pass/fail outputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.